In [2]:
import pandas as pd
import numpy as np
import surprise
from surprise.model_selection import train_test_split

In [3]:
ratings = pd.read_csv("C:/Users/PGCP-AI.STUDENTSDC/Downloads/Cases_Rec_Sys/ml-100k/u.data",sep="\t" ,names =['uid','iid','rating','ts'])
ratings.drop("ts",axis=1,inplace=True)
ratings.head()

,uid,iid,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [4]:
lowest_rating = ratings['rating'].min()
highest_rating = ratings["rating"].max()
lowest_rating, highest_rating

(np.int64(1), np.int64(5))

In [5]:
reader = surprise.Reader(rating_scale = (lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(ratings,reader)
type(data)

similarity_options  = {"name":"cosine","user_based":True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset())



Computing the cosine similarity matrix...
Done computing similarity matrix.


In [6]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold
from surprise.model_selection import train_test_split




In [7]:


trainset, testset = train_test_split(data, test_size=0.25, random_state=26)

similarity_options = {'name':'cosine','user_based':True}
algo = surprise.KNNBasic(sim_options = similarity_options, k=70) # default k=40
output = algo.fit(trainset)
pred = algo.test(testset)
surprise.accuracy.rmse(pred)


param_grid = {"k":[25,40,50,65,75], 'user_based':[False,True]}
kfold = KFold(n_splits=5, random_state=26, shuffle=True)
gs = GridSearchCV(surprise.KNNBasic,param_grid,measures = ["rmse","mae"],cv=kfold)
gs.fit(data)


Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 1.0253
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix

In [8]:
print(gs.best_score["rmse"])
print(gs.best_params["rmse"])


0.9761915086082429
{'k': 25, 'user_based': False}


In [9]:
# Building bets model on whole data
algo =gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


In [11]:
iids = ratings["iid"].unique()
iids

array([ 242,  302,  377, ..., 1637, 1630, 1641], shape=(1682,))

In [12]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)

array([2, 5, 8])

In [14]:
user = 50
u_iid = ratings[ratings['uid']==user]['iid'].unique()
print("List of items rated by user : \n",u_iid)
print("\nNo of items rated by user {0} : {1} ".format(user, len(u_iid)))
iids_ro_pred = np.setdiff1d(iids, u_iid)
print("\nItems not rated by the user or those items for which the expected ratings are to be predicted : ", iids_ro_pred)

List of items rated by user : 
 [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]

No of items rated by user 50 : 24 

Items not rated by the user or those items for which the expected ratings are to be predicted :  [   1    2    3 ... 1680 1681 1682]


In [15]:
testset = [[user,iid,0.] for iid in iids_ro_pred]
predictions = algo.test(testset)

exp_ratings = [(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
exp_ratings = pd.DataFrame(exp_ratings,columns=['iid','est_rating'])
exp_ratings.sort_values(by='est_rating',ascending=False).head(30)

,iid,est_rating
1164,1189,5.000000
1176,1201,5.000000
1097,1122,5.000000
793,814,5.000000
1475,1500,5.000000
1574,1599,5.000000
1511,1536,5.000000
1442,1467,5.000000
1628,1653,5.000000
1268,1293,5.000000


In [19]:
movies = pd.read_csv("C:/Users/PGCP-AI.STUDENTSDC/Downloads/Cases_Rec_Sys/ml-100k/movies_list.csv",encoding='latin1')
movies.head()

,movie id,movie title,release date,video release date,IMDb URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-95,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [20]:
exp_rating = exp_ratings.merge(movies[['movie id ',' movie title ',' release date ']],how='left',left_on='iid',right_on="movie id ")
exp_ratings.sort_values(by="est_rating",ascending=False).head(10)

,iid,est_rating
1164,1189,5.0
1176,1201,5.0
1097,1122,5.0
793,814,5.0
1475,1500,5.0
1574,1599,5.0
1511,1536,5.0
1442,1467,5.0
1628,1653,5.0
1268,1293,5.0
